In [3]:
# ============================================================================
# INCREMENTAL VENDOR SCRAPER - REAL-TIME SYNC
# Checks Bedrock for vendor changes and syncs with local database
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import sqlite3
from datetime import datetime
import time

# ============================================================================
# CONFIGURATION
# ============================================================================
EMAIL = 'Ramaalfaran82@gmail.com'
PASSWORD = 'Ramaalfaran82@gmail.com'
LOGIN_URL = 'https://qa.mybedrock.com/admin'
VENDOR_MASTER_URL = 'https://qa.mybedrock.com/admin/clients/vendor_master'
DB_PATH = '/content/drive/MyDrive/vendor-analysis-project/vendors.db'

# ============================================================================
# INCREMENTAL SYNC ENGINE
# ============================================================================
class IncrementalVendorSync:
    def __init__(self):
        self.driver = None
        self.db_conn = sqlite3.connect(DB_PATH)

    def initialize_driver(self):
        """Setup headless Chrome"""
        chrome_options = Options()
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        self.driver = webdriver.Chrome(options=chrome_options)
        print("✅ WebDriver initialized")

    def login(self):
        """Login to Bedrock"""
        self.driver.get(LOGIN_URL)
        time.sleep(3)
        self.driver.find_element(By.ID, 'email').send_keys(EMAIL)
        self.driver.find_element(By.ID, 'password').send_keys(PASSWORD)
        self.driver.find_element(By.CSS_SELECTOR, 'button.btn.btn-info.btn-block').click()
        time.sleep(5)
        print("✅ Login successful")

    def get_live_vendor_ids(self):
        """Scrape ALL vendor IDs currently in Bedrock"""
        self.driver.get(VENDOR_MASTER_URL)
        time.sleep(3)

        # Set page size to 100
        try:
            dropdown = self.driver.find_element(By.CSS_SELECTOR, 'select.form-control')
            self.driver.execute_script("arguments[0].value = '100';", dropdown)
            self.driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", dropdown)
            time.sleep(3)
        except:
            pass

        live_ids = set()
        page = 1

        while True:
            print(f"📄 Scanning page {page}...")
            rows = self.driver.find_elements(By.CSS_SELECTOR, 'tbody tr')

            for row in rows:
                try:
                    cells = row.find_elements(By.TAG_NAME, 'td')
                    if len(cells) > 1:
                        bedrock_id = cells[1].text.strip()
                        if bedrock_id:
                            live_ids.add(bedrock_id)
                except:
                    continue

            # Try next page
            try:
                next_btn = self.driver.find_element(By.CSS_SELECTOR, 'li.paginate_button.next:not(.disabled) a')
                parent = next_btn.find_element(By.XPATH, '..')
                if 'disabled' in parent.get_attribute('class'):
                    break
                self.driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(3)
                page += 1
            except:
                break

        print(f"✅ Found {len(live_ids)} live vendors")
        return live_ids

    def get_local_vendor_ids(self):
        """Get all vendor IDs in local database"""
        query = "SELECT bedrock_id FROM vendors WHERE bedrock_id != ''"
        df = pd.read_sql_query(query, self.db_conn)
        return set(df['bedrock_id'].astype(str))

    def sync_vendors(self):
        """Compare live vs local and mark deleted vendors"""
        print("\n🔄 STARTING SYNC PROCESS\n")

        # Get both sets
        live_ids = self.get_live_vendor_ids()
        local_ids = self.get_local_vendor_ids()

        # Find differences
        new_vendors = live_ids - local_ids
        deleted_vendors = local_ids - live_ids

        print(f"\n📊 SYNC RESULTS:")
        print(f"   Live vendors: {len(live_ids)}")
        print(f"   Local vendors: {len(local_ids)}")
        print(f"   ➕ New vendors: {len(new_vendors)}")
        print(f"   ➖ Deleted vendors: {len(deleted_vendors)}")

        # Add 'is_deleted' column if not exists
        cursor = self.db_conn.cursor()
        try:
            cursor.execute("ALTER TABLE vendors ADD COLUMN is_deleted INTEGER DEFAULT 0")
            self.db_conn.commit()
        except:
            pass

        # Mark deleted vendors
        if deleted_vendors:
            placeholders = ','.join(['?' for _ in deleted_vendors])
            cursor.execute(f"""
                UPDATE vendors
                SET is_deleted = 1, modified_date = ?
                WHERE bedrock_id IN ({placeholders})
            """, [datetime.now()] + list(deleted_vendors))
            self.db_conn.commit()
            print(f"✅ Marked {len(deleted_vendors)} vendors as deleted")

        # Restore previously deleted vendors that are back
        restored = local_ids & live_ids
        if restored:
            placeholders = ','.join(['?' for _ in restored])
            cursor.execute(f"""
                UPDATE vendors
                SET is_deleted = 0, modified_date = ?
                WHERE bedrock_id IN ({placeholders}) AND is_deleted = 1
            """, [datetime.now()] + list(restored))
            rows_restored = cursor.rowcount
            self.db_conn.commit()
            if rows_restored > 0:
                print(f"✅ Restored {rows_restored} vendors")

        # Save sync log
        log_file = '/content/drive/MyDrive/vendor-analysis-project/logs/sync_log.csv'
        log_entry = pd.DataFrame([{
            'sync_date': datetime.now(),
            'live_count': len(live_ids),
            'local_count': len(local_ids),
            'new_vendors': len(new_vendors),
            'deleted_vendors': len(deleted_vendors)
        }])
        log_entry.to_csv(log_file, mode='a', header=not os.path.exists(log_file), index=False)

        return {
            'new': list(new_vendors),
            'deleted': list(deleted_vendors),
            'total_live': len(live_ids)
        }

    def cleanup(self):
        """Close connections"""
        if self.driver:
            self.driver.quit()
        if self.db_conn:
            self.db_conn.close()

# ============================================================================
# EXECUTE SYNC
# ============================================================================
import os
os.makedirs('/content/drive/MyDrive/vendor-analysis-project/logs', exist_ok=True)

syncer = IncrementalVendorSync()
try:
    syncer.initialize_driver()
    syncer.login()
    results = syncer.sync_vendors()

    print("\n" + "="*80)
    print("✅ SYNC COMPLETE!")
    print("="*80)
    print(f"\n📊 Total live vendors: {results['total_live']}")
    print(f"➕ New vendors to scrape: {len(results['new'])}")
    print(f"➖ Vendors removed: {len(results['deleted'])}")

finally:
    syncer.cleanup()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ WebDriver initialized
✅ Login successful

🔄 STARTING SYNC PROCESS

📄 Scanning page 1...
📄 Scanning page 2...
📄 Scanning page 3...
📄 Scanning page 4...
📄 Scanning page 5...
📄 Scanning page 6...
📄 Scanning page 7...
📄 Scanning page 8...
📄 Scanning page 9...
📄 Scanning page 10...
📄 Scanning page 11...
📄 Scanning page 12...
📄 Scanning page 13...
📄 Scanning page 14...
📄 Scanning page 15...
📄 Scanning page 16...
📄 Scanning page 17...
📄 Scanning page 18...
📄 Scanning page 19...
📄 Scanning page 20...
📄 Scanning page 21...
📄 Scanning page 22...
📄 Scanning page 23...
📄 Scanning page 24...
📄 Scanning page 25...
📄 Scanning page 26...
📄 Scanning page 27...
📄 Scanning page 28...
📄 Scanning page 29...
📄 Scanning page 30...
📄 Scanning page 31...
📄 Scanning page 32...
📄 Scanning page 33...
📄 Scanning page 34...
📄 Scanning page 35...
📄 Scanning page 36...
📄 Scanning page 37.

/tmp/ipython-input-2736964005.py:149: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute(f"""


In [2]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 38.0 MB/s eta 0:00:00
